# CAR Lab：一例真实记录的 COx 学习路径

成人公开数据 · 教学用途。默认严格运行可能无有效窗口，这是应保留的结果。本笔记显式另建24/30探索运行，不能将其称为严格验证通过。所有正式计算复用 backend/car_core。

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'backend').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from backend.app.services import vitaldb, runs
from backend.car_core.config import AnalysisConfig
from backend.car_core.analysis import explain_window
from backend.app.services.export import export_zip
from scipy.stats import pearsonr
import pandas as pd
from IPython.display import display, HTML


## 1. 动态索引与真实来源
首次执行会联网下载目录与所选数值通道；缓存后全部计算可以离线进行。示例病例251来自本次实际筛查，非算法硬编码。

In [ ]:
if not vitaldb.list_cases(): vitaldb.refresh_catalog()
catalog = vitaldb.list_cases()
print('动态候选数:', len(catalog))
caseid = 251
vitaldb.download_case(caseid)
tracks, manifest = vitaldb.load_tracks(caseid)
display(pd.DataFrame([{k:manifest['case'].get(k) for k in ['caseid','age','opname','opstart','opend']}]))
display(pd.DataFrame({k:{'名义间隔':v['nominal_interval'], '实测间隔':v['interval_seconds'], 'SHA256':v['sha256']} for k,v in manifest['tracks'].items()}).T)

## 2. 原始时间戳
不按行号合并，不把上一次值无限保持。观察脑氧约0.27s短间隔与约9.8s长间隔。

In [ ]:
display(tracks['left'].head(12))
display(tracks['map'].head(12))

## 3. 严格时间块与窗口
以病例0为锚点。下表包含均值、覆盖、最长未覆盖及原因。严格30/30不足时保留null。

In [ ]:
strict = runs.run_analysis(caseid, AnalysisConfig())
display(pd.DataFrame(strict['summary']).T)
display(pd.json_normalize(strict['blocks'][200:205]))

## 4. 明确开启敏感性模式
块覆盖门槛仍0.80；仅将固定300s窗口有效配对改为至少24/30。该运行是探索，不是默认结果。

In [ ]:
exploratory = runs.run_analysis(caseid, AnalysisConfig(sensitivity=True, min_pair_ratio=.8))
display(pd.DataFrame(exploratory['summary']).T)
w = next(w for w in exploratory['windows'] if w['side']=='left' and w['cox'] is not None)
explanation = explain_window(exploratory,w['window_id'])
print(explanation['interpretation'])
print({k:explanation[k] for k in ['start','end','cox','valid_pairs','sd_map','sd_rso2']})
pairs = pd.DataFrame(explanation['pairs'])
display(pairs)

## 5. 独立核验本窗口
SciPy仅作为独立参考检查，正式结果始终来自同一内核。不要将相关值当作诊断。

In [ ]:
valid = pairs[pairs.valid]
reference = pearsonr(valid['map'], valid['rso2']).statistic
print('独立Pearson:',reference,'绝对误差:',abs(reference-w['cox']))
assert abs(reference-w['cox']) < 1e-10
display(pd.DataFrame(exploratory['map_bins']))

## 6. 导出与复现
离线报告内嵌SVG，CSV中的null留空，清单包含输入、参数、标记与代码哈希。

In [ ]:
out = ROOT/'data'/'notebook-example.zip'
out.write_bytes(export_zip(exploratory))
print(out)
print('run_id:',exploratory['run_id'])
import zipfile, io
report = zipfile.ZipFile(io.BytesIO(export_zip(exploratory))).read('report.html').decode()
display(HTML(report))

## 解释边界
MAP不是脑血流，脑氧不是直接脑血流。氧合、通气、Hb、麻醉和氧耗均可能影响关系。成人公开数据不生成新生儿治疗阈值；输出高度重叠，不是独立样本，不计算点级p值。

数据：Lee HC等，Scientific Data 9,279(2022)，DOI 10.1038/s41597-022-01411-5；CC BY4.0。方法参考：Vik等2026，DOI 10.1177/0271678X251406519。